# BERT Base NER – Named Entity Recognition

**Model:** `dslim/bert-base-NER`  
**Task:** Token classification – Named Entity Recognition  
**Entity types:** PER (person), ORG (organization), LOC (location), MISC (miscellaneous)  
**CoNLL-2003 F1:** 91.3  
**License:** MIT  
**Marketplace price:** \$0.08/hr

This notebook shows how to:
1. Deploy the model endpoint from AWS Marketplace
2. Send text for NER inference
3. Inspect entity spans with start/end offsets and confidence scores
4. Use `aggregation_strategy=simple` for span merging

## Prerequisites

- AWS account with SageMaker execution role that has `AmazonSageMakerFullAccess`
- The model must be subscribed to in AWS Marketplace before deploying
- `boto3` and `sagemaker` Python packages installed

In [ ]:
import boto3
import sagemaker
import json
import time

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print(f"Region : {region}")
print(f"Role   : {role}")

## 1. Deploy the Endpoint

Retrieve the model package ARN from AWS Marketplace and deploy it to a real-time
SageMaker endpoint. The instance type `ml.g4dn.xlarge` is required for GPU-accelerated
inference at the \$0.08/hr tier.

In [ ]:
# Replace with the model package ARN from your AWS Marketplace subscription
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

# Endpoint configuration
ENDPOINT_NAME = "bert-base-ner-endpoint"
INSTANCE_TYPE = "ml.g4dn.xlarge"
INSTANCE_COUNT = 1

sm_client = boto3.client("sagemaker", region_name=region)

# Create model from Marketplace package
model_name = f"bert-base-ner-{int(time.time())}"
sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
)

# Create endpoint config
config_name = f"{model_name}-config"
sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": INSTANCE_COUNT,
        }
    ],
)

# Create endpoint
sm_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=config_name,
)

print(f"Endpoint '{ENDPOINT_NAME}' creation started. Waiting for InService state...")

In [ ]:
# Wait for endpoint to be InService (typically 5-10 minutes)
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Endpoint '{ENDPOINT_NAME}' is InService and ready for inference.")

## 2. Invoke the Endpoint – Basic NER

The model accepts a JSON payload with an `inputs` field (a plain string).  
It returns a list of token-level entity predictions, each with:
- `entity`: IOB2 label (e.g. `B-PER`, `I-ORG`)
- `score`: confidence score (0–1)
- `index`: token position
- `word`: the token string
- `start` / `end`: character offsets in the original input

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

def predict_ner(text: str, endpoint_name: str = ENDPOINT_NAME) -> list:
    """Run NER on a single input string and return raw token predictions."""
    payload = json.dumps({"inputs": text})
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=payload,
    )
    return json.loads(response["Body"].read().decode("utf-8"))


sample_text = (
    "Hugging Face was founded in New York by Clement Delangue and "
    "Julien Chaumond. The company is backed by Google and Amazon."
)

raw_entities = predict_ner(sample_text)
print(json.dumps(raw_entities, indent=2))

## 3. Display Entity Spans with Offsets and Confidence Scores

Each prediction includes `start` and `end` character offsets into the original
input. We use these to extract the exact span text and display it alongside the
entity type and confidence score.

In [ ]:
def display_spans(text: str, entities: list) -> None:
    """Print entity spans extracted from raw token-level NER output."""
    print(f"Input: {text!r}\n")
    print(f"{'Entity Type':<12} {'Span Text':<30} {'Start':>6} {'End':>5} {'Score':>7}")
    print("-" * 65)
    for ent in entities:
        label = ent.get("entity", ent.get("entity_group", ""))
        start = ent["start"]
        end = ent["end"]
        span_text = text[start:end]
        score = ent["score"]
        print(f"{label:<12} {span_text:<30} {start:>6} {end:>5} {score:>7.4f}")


display_spans(sample_text, raw_entities)

## 4. Aggregation Strategy – Span Merging with `aggregation_strategy=simple`

By default the model returns **token-level** IOB2 labels (`B-PER`, `I-PER`, …).
Using `aggregation_strategy=simple` in the request payload instructs the
serving framework to **merge consecutive tokens** with the same entity type into
a single span.

For example, `"New"` (B-LOC) + `"York"` (I-LOC) → one span `"New York"` (LOC).

> **Note:** `aggregation_strategy` support depends on the serving container
> version. If the endpoint returns an error, omit the parameter and merge spans
> client-side using the `merge_spans()` helper below.

In [ ]:
def predict_ner_aggregated(
    text: str,
    aggregation_strategy: str = "simple",
    endpoint_name: str = ENDPOINT_NAME,
) -> list:
    """Run NER with server-side span aggregation."""
    payload = json.dumps(
        {
            "inputs": text,
            "parameters": {"aggregation_strategy": aggregation_strategy},
        }
    )
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=payload,
    )
    return json.loads(response["Body"].read().decode("utf-8"))


aggregated = predict_ner_aggregated(sample_text)
print("Aggregated spans (aggregation_strategy=simple):")
display_spans(sample_text, aggregated)

In [ ]:
def merge_spans(text: str, token_entities: list) -> list:
    """
    Client-side fallback: merge consecutive B-/I- tokens of the same entity type
    into a single span. Useful if the endpoint does not support aggregation_strategy.
    """
    merged = []
    current = None
    for ent in token_entities:
        raw_label = ent.get("entity", "")
        prefix = raw_label[:2]   # "B-" or "I-"
        etype = raw_label[2:]    # e.g. "PER", "ORG"
        if prefix == "B-" or current is None or etype != current["entity_group"]:
            if current:
                merged.append(current)
            current = {
                "entity_group": etype,
                "score": ent["score"],
                "start": ent["start"],
                "end": ent["end"],
                "word": text[ent["start"]:ent["end"]],
            }
        else:
            # Extend the current span
            current["end"] = ent["end"]
            current["word"] = text[current["start"]:current["end"]]
            current["score"] = min(current["score"], ent["score"])  # conservative
    if current:
        merged.append(current)
    return merged


# Compare raw vs. merged
merged_spans = merge_spans(sample_text, raw_entities)
print("Client-side merged spans:")
display_spans(sample_text, merged_spans)

## 5. Batch Example – Multiple Sentences

In [ ]:
sentences = [
    "Apple Inc. is headquartered in Cupertino, California.",
    "Elon Musk acquired Twitter in October 2022.",
    "The FIFA World Cup 2022 was held in Qatar.",
]

for sent in sentences:
    print("=" * 70)
    entities = predict_ner_aggregated(sent)
    display_spans(sent, entities)
    print()

## 6. Clean Up – Delete the Endpoint

Delete the endpoint when you are done to avoid ongoing charges (\$0.08/hr).

In [ ]:
sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm_client.delete_endpoint_config(EndpointConfigName=config_name)
sm_client.delete_model(ModelName=model_name)
print(f"Endpoint '{ENDPOINT_NAME}' and associated resources deleted.")